# Create a PyTorch3D wheel — environment check

Building PyTorch3D from source is the **hardest part** of the setup (CUDA toolkit,
matching torch, `nvcc`, C++ build tools). So build the wheel **first**: if this notebook
succeeds, your environment is ready for the full pipeline install. If it fails, the error
here tells you exactly what to fix — without running the whole install.

Runs anywhere with an NVIDIA GPU — **Colab, WSL2, or a Linux box**. The wheel is saved to
Google Drive on Colab, otherwise to a local folder you can find and copy.

A PyTorch3D wheel is only valid where **python (cp), torch, and CUDA all match** the
machine it was built on — so build it on the machine you'll run the pipeline on.

**WSL2 note:** you need the NVIDIA driver installed on **Windows** (not inside WSL) plus
the **CUDA toolkit installed inside WSL** (so `nvcc` exists), and a `torch` build matching
that CUDA. The preflight cell below checks all of this.

> ⚠️ Keep the session active during the build — the wheel is only written at the very end.

## 1. Install cvenv (from this repo)
Installs the copy of cvenv you were given. Falls back to GitHub if the repo isn't found.

In [1]:
import sys, subprocess, pathlib

# Prefer installing cvenv from THIS repo (the folder you were sent); else GitHub.
here = pathlib.Path.cwd()
repo = next((p for p in [here, *here.parents]
             if (p / 'pyproject.toml').exists() and (p / 'cvenv').is_dir()), None)
if repo is not None:
    print('Installing cvenv from local repo:', repo)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', str(repo)], check=True)
else:
    print('Local repo not found from cwd; installing cvenv from GitHub.')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'git+https://github.com/ribeiro-computer-vision/cvenv@v0.1.11'], check=True)

import cvenv
print('cvenv', cvenv.__version__, '| platform:', cvenv.PlatformManager().platform)

Local repo not found from cwd; installing cvenv from GitHub.
cvenv 0.1.8 | platform: Colab


## 2. Preflight: check your environment
The wheel is compiled against your installed **torch** and **CUDA toolkit**. If `torch`
is missing, no GPU is visible, or `nvcc` isn't found, fix that first — the build can't
succeed otherwise.

In [2]:
import os, shutil, subprocess

try:
    import torch
    print(f'\u2705 torch {torch.__version__} | CUDA {torch.version.cuda} | GPU available: {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        maj, mn = torch.cuda.get_device_capability(0)
        print(f'   GPU: {torch.cuda.get_device_name(0)}  (compute {maj}.{mn})')
    else:
        print('   \u26a0\ufe0f No GPU visible. A CUDA build needs a visible GPU. On WSL2 check the Windows NVIDIA driver.')
except Exception as e:
    print('\u274c torch is not importable. Install a torch build matching your CUDA first, then re-run.\n   ', e)

nvcc = shutil.which('nvcc') or '/usr/local/cuda/bin/nvcc'
if os.path.exists(nvcc):
    print('\nnvcc:', nvcc)
    subprocess.run([nvcc, '--version'])
else:
    print('\n\u274c nvcc not found. Install the CUDA toolkit (matching torch\'s CUDA) so nvcc is at '
          '/usr/local/cuda or on PATH.')

✅ torch 2.11.0+cu128 | CUDA 12.8 | GPU available: True
   GPU: NVIDIA L4  (compute 8.9)

nvcc: /usr/local/cuda/bin/nvcc


## 3. Choose where to save the wheel
**Colab** → Google Drive (survives runtime resets). **Otherwise** (WSL / local / server)
→ a local `cvenv_wheels/` folder in the current directory, easy to find and copy.

In [3]:
import os, cvenv

if cvenv.PlatformManager().platform == 'Colab':
    from google.colab import drive
    drive.mount('/content/drive')
    WHEEL_DIR = '/content/drive/MyDrive/cvenv_wheels'
else:
    WHEEL_DIR = os.path.abspath('cvenv_wheels')

os.makedirs(WHEEL_DIR, exist_ok=True)
print('Wheel will be saved to:', WHEEL_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Wheel will be saved to: /content/drive/MyDrive/cvenv_wheels


## 4. Build the wheel
Compiles PyTorch3D from source with all the fixes cvenv applies automatically: builds for
**your GPU's arch only**, sets the **CUDA-13 pulsar flag** (`-static-global-template-stub=false`),
and uses the **system linker** inside conda envs. Takes ~10–30 min.

The build is **idempotent**: if a wheel is already in `WHEEL_DIR` it is reused and nothing
is recompiled. Set `FORCE_REBUILD = True` below to rebuild instead — do that when torch,
CUDA or python has changed on this machine, since a wheel is only valid where all three
match.

### Rebuild or reuse?
The cell below reports any wheel already in `WHEEL_DIR`, checks it against **this** runtime,
and applies your choice. Reuse takes seconds; a rebuild takes ~10–30 min.

Each wheel cvenv builds gets a `.build.json` sidecar recording the python, **torch** and
**CUDA** it was compiled against — none of which appear in the filename. That is what makes
the check below trustworthy: a wheel left in Drive still installs cleanly after the runtime's
torch moves, then fails at `import pytorch3d._C`. Wheels built before this feature have no
sidecar, so they can only be reported as *unverified*.

In [7]:
# ----------------------------------------------------------------------
# False -> reuse a wheel already in WHEEL_DIR (fast; the usual case)
# True  -> recompile from source; the fresh wheel becomes the one used
FORCE_REBUILD = True
# ----------------------------------------------------------------------

import glob, os, time, cvenv

now = cvenv.components.pytorch3d.build_env()
found = sorted(glob.glob(os.path.join(WHEEL_DIR, "pytorch3d-*.whl")),
               key=os.path.getmtime, reverse=True)

print(f"this runtime: python {now['python']}  torch {now['torch']}  CUDA {now['cuda']}\n")

if not found:
    print(f"No pytorch3d wheel in {WHEEL_DIR}")
    print("The next cell will build one (~10-30 min), whatever FORCE_REBUILD says.")
else:
    print(f"{len(found)} wheel(s) already in {WHEEL_DIR}:\n")
    for w in found:
        st = os.stat(w)
        verdict, reasons = cvenv.wheel_compatibility(w)
        meta = cvenv.read_wheel_metadata(w) or {}
        mark = {True: "OK", False: "INCOMPATIBLE", None: "unverified"}[verdict]
        built = time.strftime("%Y-%m-%d %H:%M", time.localtime(st.st_mtime))
        print(f"  [{mark}] {os.path.basename(w)}")
        print(f"     {st.st_size / 1e6:.1f} MB   saved {built}")
        if meta:
            print(f"     built against python {meta.get('python')}  "
                  f"torch {meta.get('torch')}  CUDA {meta.get('cuda')}")
        for r in reasons:
            print(f"     - {r}")

    newest = found[0]
    verdict, _ = cvenv.wheel_compatibility(newest)
    print()
    if FORCE_REBUILD:
        print(f"FORCE_REBUILD = True  -> rebuilding; the new wheel supersedes "
              f"{os.path.basename(newest)}")
    elif verdict is False:
        print(f"{os.path.basename(newest)} cannot load here, so the next cell")
        print("rebuilds anyway - build_wheel() refuses to hand back a wheel it")
        print("knows will fail. Nothing to change.")
    elif verdict is None:
        print(f"FORCE_REBUILD = False -> reusing {os.path.basename(newest)}, but it is")
        print("   UNVERIFIED. If verify() fails with an _C undefined symbol, come back")
        print("   and set FORCE_REBUILD = True.")
    else:
        print(f"FORCE_REBUILD = False -> reusing {os.path.basename(newest)} (no compile)")
        print("   Set FORCE_REBUILD = True above to rebuild anyway.")

this runtime: python 3.13.15  torch 2.11.0+cu128  CUDA 12.8

1 wheel(s) already in /content/drive/MyDrive/cvenv_wheels:

  [unverified] pytorch3d-0.7.8-cp312-cp312-linux_x86_64.whl
     65.3 MB   saved 2026-07-30 20:35
     - no build metadata recorded beside this wheel

FORCE_REBUILD = True  -> rebuilding; the new wheel supersedes pytorch3d-0.7.8-cp312-cp312-linux_x86_64.whl


In [8]:
whl = cvenv.get_component('pytorch3d').build_wheel(out_dir=WHEEL_DIR, force=FORCE_REBUILD)
print('\n✅ wheel to install:', whl)

Building a PyTorch3D wheel from source (ref=stable); this can take several minutes…
$ /usr/bin/python3 -m pip install numpy>=2.0,<2.1
$ /usr/bin/python3 -m pip install iopath
$ /usr/bin/python3 -m pip install --root-user-action ignore ninja
TORCH_CUDA_ARCH_LIST = 8.9 | CUDA_HOME = /usr/local/cuda | NVCC_APPEND_FLAGS = -static-global-template-stub=false
$ /usr/bin/python3 -m pip wheel --no-deps --no-build-isolation git+https://github.com/facebookresearch/pytorch3d.git@stable -w /content/drive/MyDrive/cvenv_wheels
💾 saved reusable wheel: /content/drive/MyDrive/cvenv_wheels/pytorch3d-0.7.8-cp313-cp313-linux_x86_64.whl
   provenance recorded in pytorch3d-0.7.8-cp313-cp313-linux_x86_64.whl.build.json

✅ wheel to install: /content/drive/MyDrive/cvenv_wheels/pytorch3d-0.7.8-cp313-cp313-linux_x86_64.whl


## 5. Install & verify
`verify()` runs `import pytorch3d._C` — the real proof the compiled extension matches this
machine's torch/CUDA. If you see `✅ pytorch3d … (_C OK)`, **your environment is good to go.**

In [9]:
cvenv.get_component('pytorch3d').install(wheel_url=whl)
cvenv.get_component('pytorch3d').verify()

▶️  pytorch3d: installing…
$ /usr/bin/python3 -m pip install numpy>=2.0,<2.1
$ /usr/bin/python3 -m pip install iopath
   provenance matches: built against torch 2.11.0+cu128 / CUDA 12.8
Installing PyTorch3D from wheel: pytorch3d-0.7.8-cp313-cp313-linux_x86_64.whl
$ /usr/bin/python3 -m pip install --force-reinstall --no-deps /content/drive/MyDrive/cvenv_wheels/pytorch3d-0.7.8-cp313-cp313-linux_x86_64.whl
✅ installed from provided wheel.
✅ pytorch3d 0.7.8 (_C OK)


True

## Done — and how to reuse it
That `.whl` in `WHEEL_DIR` is a normal file. On the **same** machine/stack (same python,
torch, CUDA), skip the build next time — point `wheel_url` at it:

```python
import cvenv, glob, os
whl = max(glob.glob(os.path.join(WHEEL_DIR, 'pytorch3d-*.whl')), key=os.path.getmtime)
cvenv.get_component('pytorch3d').install(wheel_url=whl)
```

Each wheel is saved with a `.build.json` sidecar recording the python, torch and CUDA it
was compiled against. `install(wheel_url=...)` reads it and tells you up front whether the
wheel can work here — and fetches it alongside the wheel when the URL is remote. **If you
copy or share a wheel, take the sidecar too**; without it the wheel still installs, but the
check falls back to *unverified*.

If torch or CUDA later changes on this machine, `verify()` will fail with an `_C`
`undefined symbol` error — just re-run this notebook with `FORCE_REBUILD = True` to rebuild.

**If the build failed:** the message tells you what your environment is missing (no GPU,
no `nvcc`, torch/CUDA mismatch). Fix that and re-run — you don't need to attempt the full
pipeline install until this wheel builds and `verify()` passes.